# Generating type 4 clones with LLMs

## Generation

In [1]:
DATASET_PATH = "../dataset/bigcodebench_normalized.json"
OUT_PATH      = "../results/bigcodebench_llm_clones.json"
OLLAMA_MODEL = "llama3.1:latest"   

# Generation settings
LLM_OPTS = {
    "temperature": 0.6,
    "top_p": 0.95,
    "repeat_penalty": 1.05,
    "num_predict": 768,   
}

N_ENTRIES = 4
CLONES_PER_ENTRY = 2  

In [2]:
# import os, json  
# from src.clone_gen import STRATEGY_HINTS, SYSTEM_PROMPT, build_user_prompt, generate_clones

# with open(DATASET_PATH, "r", encoding="utf-8") as f:
#     data = json.load(f)

# sample = data[:N_ENTRIES]

# results = []
# for i, entry in enumerate(sample, 1):
#     print(f"\nGenerating clones for Entry {i}/{len(sample)} | id={entry['id']}")
#     clones = []

#     original_body = entry["original_code"]
#     tests_list    = entry["test"]
#     description   = entry.get("description", "")
#     libs          = entry.get("metadata", {}).get("libs", [])

#     tests_snippet = tests_list[0] if tests_list else ""

#     for k in range(CLONES_PER_ENTRY):
#         hint = STRATEGY_HINTS[k % len(STRATEGY_HINTS)]
#         user_prompt = build_user_prompt(original_body, description, libs, tests_snippet, hint)

#         messages = [
#             {"role": "system", "content": SYSTEM_PROMPT},
#             {"role": "user",   "content": user_prompt}
#         ]

#         try:
#             code = generate_clones(messages, model=OLLAMA_MODEL, options=LLM_OPTS, expected_func_name="task_func")
#             clones.append({
#                 "transformation": f"LLM/{OLLAMA_MODEL}",
#                 "strategy_hint": hint,
#                 "code": code
#             })
#         except Exception as e:
#             print(f" Error generating clone {k+1}: {e}")

#     results.append({
#         "id": entry["id"],
#         "language": entry["language"],
#         "description": description,
#         "metadata": entry.get("metadata", {}),
#         "original_code": original_body,
#         "test": tests_list,
#         "clones": clones
#     })

# os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
# with open(OUT_PATH, "w", encoding="utf-8") as f:
#     json.dump(results, f, indent=2)


## Running the tests on the clones

In [3]:
import os, json
from src.utils import validate_with_unittest

with open(OUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

results = []
for i, entry in enumerate(data, 1):
    print(f"\nTesting Entry {i}/{len(data)} | id={entry['id']}")
    clones = []

    tests_list = entry["test"]

    for k, clone in enumerate(entry.get("clones", []),1):
        try:
            code = clone["code"]
            # Get individual test results
            test_results = validate_with_unittest(code, tests_list) # this function is defined in 2.preprocess.ipynb
            clone["test_results"] = test_results

            # Print summary
            passed = sum(1 for v in test_results.values() if v=="PASS")
            total = len(test_results)
            print(f"  Clone {k}: {passed}/{total} tests passed")

            clones.append(clone)
        except Exception as e:
            print(f"  Error testing clone {k}: {e}")
            clone["test_results"] = {}
            clones.append(clone)

    entry["clones"] = clones
    results.append(entry)

# Save dataset with test results
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Done. Saved dataset with test results to {OUT_PATH}")



Testing Entry 1/4 | id=BigCodeBench/0
  Clone 1: 10/10 tests passed
  Clone 2: 10/10 tests passed

Testing Entry 2/4 | id=BigCodeBench/1
  Clone 1: 3/3 tests passed
  Clone 2: 3/3 tests passed

Testing Entry 3/4 | id=BigCodeBench/2
  Clone 1: 3/5 tests passed
  Clone 2: 5/5 tests passed

Testing Entry 4/4 | id=BigCodeBench/3
  Clone 1: 5/5 tests passed
  Clone 2: 5/5 tests passed

✅ Done. Saved dataset with test results to ../results/bigcodebench_llm_clones.json


## Calculating similarity scores

In [ ]:
import json, fractions
from collections import Counter
from nltk.util import ngrams
import numpy as np
from crystalbleu import corpus_bleu
from codebleu import calc_codebleu
from nltk.translate.bleu_score import SmoothingFunction
from src.similarity import tokenize_code, is_trivial_ngram

# === CONFIG === 
K = 500  # top-k most common ngrams to ignore
sm_func = SmoothingFunction(epsilon=0.0001).method1

# === Load dataset ===
with open(OUT_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

# === Build global ngram frequency ===
all_ngrams = []
for entry in dataset:
    tokens = tokenize_code(entry["original_code"])
    for n in range(1, 5):
        all_ngrams.extend(list(ngrams(tokens, n)))
    for clone in entry.get("clones", []):
        tokens = tokenize_code(clone["code"])
        for n in range(1, 5):
            all_ngrams.extend(list(ngrams(tokens, n)))

freq = Counter(all_ngrams)


most_common_dict = {ngram:1 for ngram,_ in freq.most_common(K) if not is_trivial_ngram(ngram)}

# === Collect intra/inter pairs ===
intra_r, intra_h = [], []
inter_r, inter_h = [], []
labels, bleu_scores, codebleu_scores, crystal_scores = [], [], [], []
# --- Patch Fraction for CrystalBLEU ---
_OrigFraction_new = fractions.Fraction.__new__ 
def fraction_new_patched(cls, *args, **kwargs):
    kwargs.pop("_normalize", None)
    return _OrigFraction_new(cls, *args, **kwargs)
fractions.Fraction.__new__ = fraction_new_patched
for entry in dataset:
    ref = tokenize_code(entry["original_code"])
    for clone in entry.get("clones", []):
        hyp = tokenize_code(clone["code"])

        # assume ALL are intra (positive)
        label = 1  
        labels.append(label)

        # BLEU
        bleu = corpus_bleu([ref], [hyp], smoothing_function=sm_func)
        # CrystalBLEU
        crystal = corpus_bleu([ref], [hyp], smoothing_function=sm_func, ignoring=most_common_dict)
        # CodeBLEU
        codebleu_score = calc_codebleu([entry["original_code"]], [clone["code"]], lang="python")["codebleu"]

        bleu_scores.append(bleu)
        codebleu_scores.append(codebleu_score)
        crystal_scores.append(crystal)

        # Save in dataset
        clone.setdefault("metrics", {})
        clone["metrics"].update({
            "bleu": float(bleu),
            "crystalbleu": float(crystal),
            "codebleu": float(codebleu_score),
        })

        intra_r.append([ref])
        intra_h.append(hyp)

# --- Restore original Fraction ---
fractions.Fraction.__new__ = _OrigFraction_new

mean_bleu = np.mean(bleu_scores)
mean_codebleu = np.mean(codebleu_scores)
mean_crystal = np.mean(crystal_scores)
print(f"Avg BLEU: {mean_bleu:.4f}")
print(f"Avg CodeBLEU: {mean_codebleu:.4f}")
print(f"Avg CrystalBLEU: {mean_crystal:.4f}")

# === Save dataset with metrics ===
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2)

print("✅ Finished scoring with BLEU, CrystalBLEU, and CodeBLEU.")


RecursionError: maximum recursion depth exceeded